> **Generated notebook — do not edit here.**  
> Source: `01_scripts/01_quality_control_saureus.Rmd`, which is also a chapter of the course book.  
> To change anything, edit the Rmd and run `python3 util/rmd_to_ipynb.py`.
>
> Run the notebooks in order — **01 → 02 → 03** — with the **R** kernel; each step saves results that the next one loads.

In [ ]:
# Match the report's defaults: warnings hidden (warning=FALSE in the Rmd)
# and 7 x 5 inch figures. Remove the warn option to see warnings.
options(warn = -1, repr.plot.width = 7, repr.plot.height = 5)

# Quality Control

This report covers the quality control and exploratory analysis of bulk RNA-seq data from *Staphylococcus aureus* grown as biofilm versus planktonic cultures, from [Tomlinson *et al.*, 2021](https://doi.org/10.1099/mgen.0.000598) (GEO: GSE163153, PRJNA685119).

The study profiled five MRSA clonal lineages (USA100–USA500) as biofilm and planktonic cultures at three time points (5 h, 10 h, 24 h). This workshop uses **two strains, analysed independently**: **USA100** (N315 reference) and **USA500** (aligned to the USA300 reference). Each strain is a self-contained analysis, following the paper's own per-strain design.

The contrast of interest is:

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

- Biofilm vs planktonic (control), **within a single strain and time point**

</div>

**Topics covered**

-   Exploratory data analyses
-   Sanity Checks
-   Choosing a time point for downstream differential expression

**Data**

-   Reference genome: strain-specific. **USA100** → N315 (`GCF_000009645.1`); **USA500** → USA300 (`GCF_000013465.1`). The USA500 isolate has no finished reference of its own, so USA300 (its closest relative) was used, as in the paper.

-   Input data is the output from the [nf-core/rnaseq pipeline](https://nf-co.re/rnaseq/) `salmon.merged.gene.SummarizedExperiment.rds`, run with the **prokaryotic profile** (Bowtie2 + Salmon).

<div style="background:#eaf4fd;border-left:5px solid #3498db;border-radius:4px;padding:10px 16px;margin:10px 0;">

<strong>📘 Note:</strong> Key points about this dataset:
<ul>
<li>Reads are aligned with a non-splice-aware aligner (Bowtie2), appropriate for bacterial genomes with no introns</li>
<li>The prokaryotic reference GTF carries no gene symbols, so genes are labelled by their <strong>locus tags</strong> (e.g. <code>SA_RS00005</code> for N315, <code>SAUSA300_RS00010</code> for USA300). No symbol conversion is possible or needed at this stage</li>
<li>Because the two strains use different reference genomes, their gene identifiers are not comparable — the analyses are kept fully separate and are never merged</li>
<li>Enrichment analysis (a later script) uses KEGG for <em>S. aureus</em>; g:Profiler does not support this organism</li>
</ul>

</div>

<div style="background:#eaf4fd;border-left:5px solid #3498db;border-radius:4px;padding:10px 16px;margin:10px 0;">

<strong>📘 Note on this experiment:</strong> The full dataset for each strain contains 32 samples: biofilm and
planktonic cultures at 5 h (n = 4 each), 10 h (n = 4 each) and 24 h (n = 8 each). This QC script loads
<strong>all 32 samples</strong> so the PCA can reveal the time-point structure. The choice of which time
point to carry into differential expression is made <em>after</em> looking at this QC, not before.

</div>

**Explanation of the QC analysis 🧾**

This document guides you through the standard QC pipeline for bulk RNA-seq data processed with [nf-core/rnaseq](https://nf-co.re/rnaseq/) (`Bowtie2` + `Salmon`, prokaryotic profile). It is structured to be both educational and reproducible.

<div style="background:#eaf4fd;border-left:5px solid #3498db;padding:0.5em 1em;margin:1em 0;border-radius:6px;">

<strong>⭐ One file, two strains.</strong> Set the <code>strain</code> variable in the first code chunk to
either <code>"USA-100"</code> or <code>"USA-500"</code> and render. Every path, label and output folder
below is derived from that one setting, so the same script produces the QC for either strain without
further edits.

</div>

**Setup the Environment**

<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px;margin:10px 0;">

<strong>💡 Tip:</strong> Install packages only once!

</div>

<div style="background:#d1ecf1;border-left:4px solid #0c5460;padding:10px;margin:10px 0;">

<strong>📌 Remember:</strong> Load all libraries at the start of every session before running any analysis.

</div>

In [ ]:
library(tidyverse)
library(reshape2)
library(DESeq2)
library(ggpubr)
library(RColorBrewer)
library(pheatmap)
library(factoextra)
library(knitr)
library(kableExtra)
library(DT)

## Choose the Strain

Everything downstream keys off this single setting. Change it to `"USA-500"` to run the other strain.

In [ ]:
# --- The only line you need to change to switch strains ---
strain <- "USA-100"          # "USA-100" or "USA-500"
# ----------------------------------------------------------

stopifnot(strain %in% c("USA-100", "USA-500"))

# A filesystem-safe tag used for the results sub-folder (e.g. "usa100")
strain_tag <- tolower(gsub("-", "", strain))

git_root    <- system("git rev-parse --show-toplevel", intern = TRUE)
data_dir    <- file.path(git_root, "data", "data-01-Staphylococcus_aureus", strain)
results_dir <- file.path(git_root, "results", strain_tag)

cat("Strain        :", strain, "\n")
cat("Data folder   :", data_dir, "\n")
cat("Results folder:", results_dir, "\n")

### Loading Count Data

The nf-core/rnaseq pipeline (prokaryotic profile: `Bowtie2` + `Salmon`) produces a `SummarizedExperiment` object. We load it, extract the raw count matrix, and use the locus tags as row names.

<div style="background:#fdf2e9;border-left:5px solid #e67e22;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>⭐ Important:</strong> Salmon returns estimated counts, which are not integers. DESeq2 requires integer counts, so we round (not truncate) before building the object. This matches the tximport convention used by nf-core/differentialabundance.

</div>

<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px;margin:10px 0;">

<strong>💡 Tip:</strong> If you are unsure which assay name or rowData columns are available in your RDS, inspect them first with <code>assayNames(count_x)</code> and <code>names(rowData(count_x))</code>.

</div>

In [ ]:
count_x <- readRDS(
  file.path(data_dir, "salmon.merged.gene.SummarizedExperiment.rds")
)

# Inspect what is available (uncomment to explore):
# assayNames(count_x); names(rowData(count_x))

count_genes <- assay(count_x, assayNames(count_x)[1])

gene_ids   <- rowData(count_x)$gene_id
gene_names <- rowData(count_x)$gene_name

# Salmon estimates are fractional -> round to nearest integer for DESeq2.
# assay() can return a data.frame, which breaks storage.mode<- ; force a numeric matrix first.
count_genes <- as.matrix(count_genes)
mode(count_genes) <- "numeric"
count_genes <- round(count_genes)
storage.mode(count_genes) <- "integer"

# Row names are the locus tags (gene_id). In this prokaryotic GTF gene_name == gene_id,
# so there is nothing to swap to; make.unique() guards against any accidental duplicates.
rownames(count_genes) <- make.unique(as.character(gene_ids))

cat("Dimensions (genes × samples):", dim(count_genes), "\n")
print(head(rownames(count_genes)))

<div style="background:#eaf4fd;border-left:5px solid #3498db;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>📘 Note:</strong> This prokaryotic reference has no gene symbols, so <code>gene_id</code> and
  <code>gene_name</code> are the same locus tags. We keep the locus tags as row names throughout. This is
  expected for bacterial nf-core runs and is not a problem — locus tags are the stable identifiers used by
  KEGG for enrichment later.

</div>

In [ ]:
# Confirm the two ID columns are identical (they are, for this GTF), and that
# the IDs look like S. aureus locus tags rather than symbols.
cat("gene_id == gene_name for all rows? :",
    identical(as.character(gene_ids), as.character(gene_names)), "\n")
cat("Example locus tags                 :",
    paste(head(gene_ids, 3), collapse = ", "), "\n")

## Building the Metadata from Sample Names

The sample names encode the full experimental design in a fixed pattern:

`{STRAIN}_{REF}_{LIFESTYLE}_{TIMEPOINT}h_{REPLICATE}` — e.g. `USA100_N315_Biofilm_24h_5`.

Rather than maintain a separate metadata file that could drift out of sync with the count matrix, we parse the design directly from the column names. This guarantees the metadata always matches the samples actually present in the data.

In [ ]:
samples_info <- tibble(sample = colnames(count_genes)) %>%
  tidyr::separate(
    sample,
    into   = c("strain_id", "ref", "lifestyle", "timepoint", "replicate"),
    sep    = "_",
    remove = FALSE
  ) %>%
  mutate(
    lifestyle = factor(lifestyle, levels = c("Planktonic", "Biofilm")),  # Planktonic = reference
    timepoint = factor(timepoint, levels = c("5h", "10h", "24h")),
    replicate = as.integer(replicate)
  )

# Sanity: every column parsed into exactly the expected fields
stopifnot(!any(is.na(samples_info$lifestyle)),
          !any(is.na(samples_info$timepoint)))

cat("Samples parsed:", nrow(samples_info), "\n\n")
print(table(samples_info$lifestyle, samples_info$timepoint))

<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px;margin:10px 0;">

<strong>💡 Tip:</strong> The table above is your design at a glance. Note it is <strong>unbalanced across
time</strong>: 5 h and 10 h have 4 replicates per lifestyle, while 24 h has 8. This does not affect QC, but
keep it in mind when choosing a time point for differential expression.

</div>

In [ ]:
DT::datatable(
  data       = samples_info,
  rownames   = FALSE,
  extensions = c('Buttons', 'Scroller'),
  options    = list(
    dom         = 'Bfrtip',
    buttons     = c('copy', 'csv'),
    deferRender = TRUE,
    scrollX     = TRUE,
    scrollY     = 200,
    scroller    = TRUE
  ),
  caption = paste0('Sample metadata — ', strain, ' (parsed from sample names)')
)

## Preparing the Data

Before building the `DESeqDataSet` we:

1.  Set factor levels so that `Planktonic` is the **reference level** (the denominator in fold-change calculations — biofilm is compared *to* planktonic).
2.  Align the sample order between the count matrix and the metadata — `DESeq2` requires these to match exactly.

In [ ]:
# Order metadata to match the count matrix columns exactly
samples_info <- samples_info[match(colnames(count_genes), samples_info$sample), ]
samples_info <- as.data.frame(samples_info)
rownames(samples_info) <- samples_info$sample

stopifnot(identical(colnames(count_genes), samples_info$sample))

print(data.frame(count_col = colnames(count_genes),
                 lifestyle = samples_info$lifestyle,
                 timepoint = samples_info$timepoint))

### Creating the DESeqDataSet

The `DESeqDataSet` (DDS) holds the raw count matrix, sample metadata, and a design formula.

<div style="background:#eaf4fd;border-left:5px solid #3498db;border-radius:4px;padding:10px 16px;margin:10px 0;">

<strong>📘 Note:</strong> **Design at the QC stage.** For quality control we only need a valid object to
hold the counts and to compute a variance-stabilising transformation; the design is set to
`~ lifestyle` as a sensible placeholder. The QC transformation below uses `blind = TRUE`, so it ignores
the design entirely and gives an unbiased view of sample similarity. The design that actually matters is
set in the differential expression script, once a time point has been chosen.

</div>

In [ ]:
dds <- DESeqDataSetFromMatrix(
  countData = count_genes,
  colData   = samples_info,
  design    = ~ lifestyle
)

### Sanity Checks

<div style="background:#d1ecf1;border-left:4px solid #0c5460;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>📌 Remember:</strong> Always do a sanity check!

</div>

**Are We Working with Raw Counts?**

`DESeq2` requires **raw, un-normalised integer counts**. Feeding it normalised values (TPM, FPKM) will produce incorrect results.

In [ ]:
options(scipen = 999)

kable(count_genes[1:6, 1:min(8, ncol(count_genes))],
      caption = "Raw count matrix — first 6 genes, first samples",
      format.args = list(big.mark = ",")) %>%
  kable_styling(bootstrap_options = c("striped", "hover", "condensed"),
                full_width = FALSE)

barplot(colSums(count_genes),
        main   = paste0("Library sizes (total counts per sample) — ", strain),
        ylab   = "Total raw counts",
        xlab   = NULL,
        col    = "steelblue",
        las    = 2,
        cex.names = 0.5,
        names.arg = colnames(count_genes))

<div style="background:#fff3cd;border-left:4px solid #ffc107;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>💡 Tip:</strong> scipen = 999 is a penalty against scientific notation. R uses it to decide when to switch between fixed (150000) and scientific (1.5e+05) format. The default is scipen = 0 — by setting it to 999 you make the penalty so high that R almost never switches to scientific notation, preferring plain numbers instead.

</div>

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

A bacterial genome has far fewer genes than a human one (roughly 2,500–2,900 for *S. aureus*), so this count matrix is much smaller than a eukaryotic one. A right-skewed count distribution (many low-count genes, a few very high) is still expected and correct.

</div>

### Pre-filtering Low-count Genes

Genes with very few counts across all samples carry no statistical power and inflate the multiple testing burden. We remove genes that do not have at least 10 counts in a minimum number of samples (equal to the size of the smallest lifestyle group).

Unlike the human genome, most bacterial genes are expressed under standard growth, so expect only a modest fraction to be filtered out.

In [ ]:
smallestGroupSize <- min(table(samples_info$lifestyle))

cat("Smallest lifestyle group size:", smallestGroupSize, "\n")
cat("Filtering threshold          : at least 10 counts in", smallestGroupSize, "or more samples\n")

keep <- rowSums(counts(dds) >= 10) >= smallestGroupSize
n_before <- nrow(dds)
dds  <- dds[keep, ]

cat("Genes before filtering:", n_before, "\n")
cat("Genes after  filtering:", nrow(dds), "\n")
cat("Genes removed         :", sum(!keep), "\n")

### Factor Order and Reference Level

The first factor level is always the reference (denominator) in `DESeq2` comparisons. Setting `Planktonic` as the reference ensures fold changes are computed in the intended direction: **biofilm vs planktonic**, not the reverse.

In [ ]:
dds$lifestyle <- relevel(dds$lifestyle, ref = "Planktonic")
levels(dds$lifestyle)

<div style="background:#d4edda;border-left:4px solid #28a745;padding:10px;margin:10px 0;">

<strong>📌 Remember:</strong> The reference level determines the direction of fold changes. A positive log2FC will mean higher expression in biofilm relative to planktonic.

</div>

## Exploratory Data Analysis

In this section we do **not** perform statistical tests. The goal is **quality control:** check count distributions, detect technical outliers, and confirm that samples cluster as expected — and, specifically for this dataset, to see how strongly time point structures the data.

### Estimate Size Factors

Library sizes differ between samples due to technical variation in sequencing depth. `DESeq2`'s median-of-ratios normalisation corrects for this with a size factor per sample. Values close to 1.0 indicate balanced libraries; values far from 1.0 warrant investigation.

In [ ]:
dds <- estimateSizeFactors(dds)

sizeFactors(dds) %>%
  enframe(name = "sample", value = "size_factor") %>%
  kable(digits = 3, caption = "DESeq2 size factors per sample") %>%
  kable_styling(bootstrap_options = c("striped", "hover"), full_width = FALSE)

### Distribution of Normalised Counts

Boxplots of log2-normalised counts per sample check that all samples have comparable expression distributions. After normalisation, boxes should overlap substantially. A sample that is a clear outlier in median or spread may indicate a failed library or mislabelled sample.

<div style="background:#f8d7da;border-left:4px solid #721c24;padding:10px;margin:10px 0;">

<strong>📌 Remember:</strong> These normalised counts are for visualisation only. Always feed DESeq2 <strong>raw integer counts</strong>.

</div>

In [ ]:
normalized_counts <- counts(dds, normalized = TRUE)

counts_norm <- reshape2::melt(
  normalized_counts,
  varnames   = c("gene_id", "sample"),
  value.name = "counts"
)

counts_norm <- inner_join(
  counts_norm,
  as.data.frame(colData(dds)),
  by = "sample"
)

dir.create(file.path(results_dir, "plots"), recursive = TRUE, showWarnings = FALSE)

distribution <- ggplot(counts_norm,
                       aes(x    = sample,
                           y    = log2(counts + 1),
                           fill = lifestyle)) +
  geom_boxplot(outlier.size = 0.3, alpha = 0.8) +
  coord_flip() +
  theme_pubr(border = TRUE) +
  xlab("Sample") +
  ylab("log2(normalised counts + 1)") +
  ggtitle(paste0("Normalised count distribution — ", strain))

distribution

ggsave(
  filename = file.path(results_dir, "plots", "normalised_count_distribution.png"),
  plot     = distribution,
  width    = 8,
  height   = 8,
  dpi      = 300
)

### Sample Correlation Heatmap

Euclidean distances between `VST-transformed` samples reveal how similar samples are to each other globally. Samples from the same lifestyle, or the same time point, may cluster together.

`Variance-stabilising transformation (VST)` is applied because raw or normalised counts have heteroscedastic variance (high-count genes have much larger absolute variance).

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

VST removes this mean–variance dependence, making distances meaningful across the full expression range ([Anders & Huber, 2010](https://doi.org/10.1186/gb-2010-11-10-r106)).

</div>

In [ ]:
vsd <- varianceStabilizingTransformation(dds, blind = TRUE)

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

With **`blind = TRUE`**, the VST is computed ignoring the experimental design. This is recommended for QC and exploratory analysis, where you want an unbiased view of sample similarity.

Use `blind = FALSE` only when the transformation is feeding into a model that already accounts for the design.

</div>

In [ ]:
sampleDists      <- dist(t(assay(vsd)))
sampleDistMatrix <- as.matrix(sampleDists)

rownames(sampleDistMatrix) <- paste(vsd$lifestyle, vsd$timepoint, vsd$replicate, sep = " | ")
colnames(sampleDistMatrix) <- rownames(sampleDistMatrix)

heat <- pheatmap(
  sampleDistMatrix,
  main     = paste0("Sample-to-sample distances (VST) — ", strain),
  fontsize = 7
)

heat

png(
  filename = file.path(results_dir, "plots", "sample_distance_heatmap.png"),
  width    = 9,
  height   = 9,
  units    = "in",
  res      = 300
)
heat
dev.off()

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

The diagonal is always zero (a sample compared to itself). Dark = similar, light = distant. Look at whether samples group by lifestyle, by time point, or both. If time point is a strong driver of the distances, that is a signal that the time points should be analysed separately rather than pooled — which is exactly why we pick a single time point for differential expression.

</div>

### Principal Component Analysis (PCA)

PCA reduces the high-dimensional expression space (one dimension per gene) to a few principal components capturing the largest sources of variance. Here it answers two questions at once: does lifestyle (biofilm vs planktonic) separate the samples, and how much of the structure is driven by time point?

In [ ]:
pca_data <- plotPCA(vsd,
                    intgroup   = c("lifestyle", "timepoint"),
                    returnData = TRUE)

pct_var <- round(100 * attr(pca_data, "percentVar"), 1)

PCAPlot <- ggplot(pca_data, aes(x     = PC1,
                                y     = PC2,
                                color = lifestyle,
                                shape = timepoint)) +
  geom_point(size = 4, alpha = 0.85) +
  geom_hline(yintercept = 0, linetype = "dashed", alpha = 0.3) +
  geom_vline(xintercept = 0, linetype = "dashed", alpha = 0.3) +
  theme_pubr(border = TRUE) +
  theme(
    axis.text       = element_text(size = 12),
    axis.title      = element_text(size = 14),
    legend.text     = element_text(size = 12),
    legend.position = "bottom"
  ) +
  labs(
    x     = paste0("PC1: ", pct_var[1], "% variance"),
    y     = paste0("PC2: ", pct_var[2], "% variance"),
    title = paste0("PCA — ", strain, " (VST-transformed counts)"),
    color = "Lifestyle",
    shape = "Time point"
  )

PCAPlot

ggsave(
  filename = file.path(results_dir, "plots", "PCA_Plot.png"),
  plot     = PCAPlot,
  width    = 8,
  height   = 6,
  dpi      = 300
)

<div style="background:#eaf4fd;border-left:5px solid #3498db;padding:0.5em 1em;margin:1em 0;border-radius:6px;">

<strong>⭐ Using the PCA to choose a time point.</strong> This is the decision the QC is built around.
Look at which axis separates <strong>lifestyle</strong> (colour) and which separates <strong>time point</strong>
(shape). The paper reports that differential expression between biofilm and planktonic grows with culture
age, being strongest at 24 h. A good time point for the downstream simple two-group comparison is one where
biofilm and planktonic separate cleanly <em>and</em> where you have adequate replication. Read the actual
separation off this plot rather than assuming it — then set that time point in script 02.

</div>

### Detecting Outliers

The PCA above uses only the top 500 most variable genes (DESeq2 default). Here we run PCA on the full VST matrix and inspect a scree plot and biplot to assess whether any single sample drives an unusual amount of variance, a common sign of a technical outlier.

In [ ]:
pca_full <- prcomp(t(assay(vsd)))

screeplot <- fviz_screeplot(pca_full, addlabels = TRUE,
               main = paste0("Scree plot — variance per PC (", strain, ")"))
screeplot

pca_ind <- fviz_pca_ind(pca_full, geom = c("point"), repel = TRUE,
             habillage = samples_info$lifestyle,
             title = paste0("PCA — sample positions, full gene matrix (", strain, ")"))
pca_ind

ggsave(
  filename = file.path(results_dir, "plots", "pca_screeplot.png"),
  plot     = screeplot, width = 8, height = 6, dpi = 300
)
ggsave(
  filename = file.path(results_dir, "plots", "pca_individuals.png"),
  plot     = pca_ind, width = 8, height = 6, dpi = 300
)

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

**What to look for:** an isolated sample sitting far from every other point on the individuals plot, or a single sample dominating PC1 on the scree plot, is a candidate technical outlier. Investigate its library size and its MultiQC metrics before deciding whether to exclude it.

</div>

### Top Variable Genes Heatmap

Clustering the 50 most variable genes across samples gives a gene-level view of the structure in the data. Row-scaling (z-score per gene) is applied so that highly expressed genes do not visually dominate.

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 11)  # figure size set in the Rmd chunk
topVarGenes <- head(order(rowVars(assay(vsd)), decreasing = TRUE), 50)

df_anno <- as.data.frame(colData(vsd)[, c("lifestyle", "timepoint"), drop = FALSE])

heat_plot <- pheatmap(
  assay(vsd)[topVarGenes, ],
  cluster_cols             = TRUE,
  cluster_rows             = TRUE,
  scale                    = "row",
  clustering_distance_rows = "euclidean",
  clustering_distance_cols = "euclidean",
  annotation_col           = df_anno,
  show_colnames            = FALSE,
  show_rownames            = TRUE,
  fontsize_row             = 7,
  main                     = paste0("Top 50 variable genes — ", strain, " (VST, row-scaled)")
)

heat_plot

png(
  filename = file.path(results_dir, "plots", "heatmap_top50_variable.png"),
  width    = 9,
  height   = 11,
  units    = "in",
  res      = 300
)
heat_plot
dev.off()

<div style="background:#eaf4fd;border-left:5px solid #3498db;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>📘 Note:</strong> Rows are locus tags (this prokaryotic reference has no gene symbols). They are
  the same identifiers KEGG uses, so they can be carried directly into the enrichment analysis with no
  conversion step.

</div>

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

**How to read this heatmap:**

- Each **row** is a gene, each **column** is a sample
- Colours show the **z-score** (row-scaled expression) — red = higher than average for that gene, blue = lower
- The **dendrograms** show hierarchical clustering — samples/genes that behave similarly are grouped together

**What to look for:**

- Clear colour blocks by lifestyle → biofilm vs planktonic has a strong, consistent transcriptional effect
- Whether samples split by time point first (a sign the time points are very different states)

</div>

## Summary

Before proceeding to differential expression, confirm the QC checks and record the time-point decision:

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

| Check | What to confirm |
|---|---|
| Size factors ≈ 1.0 across samples | Library sizes are balanced |
| Boxplots of normalised counts overlap | No extreme outlier samples |
| Correlation heatmap | Note whether structure is by lifestyle, by time point, or both |
| PCA | Note which axis is lifestyle and which is time point |
| No isolated samples | No technical outliers |
| **Time-point choice** | **Record which time point you will take into script 02, and why** |

</div>

<div style="font-size: 0.9em; color: grey;">

*This QC loads all three time points so the structure is visible. Differential expression (script 02) uses
a single time point with the simple design `~ lifestyle` (biofilm vs planktonic), following the paper's
per-strain comparison. The two strains are never merged: they use different reference genomes and their
locus tags are not comparable.*

</div>

<div style="background:#fdf2e9;border-left:5px solid #e67e22;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>⭐ Important:</strong> If any check fails, investigate the cause before running DESeq2. Proceeding with outlier samples or a mis-specified design will compromise all downstream results.

</div>

In [ ]:
# ── Export the full (all-timepoints) DDS for reference ────────────────────────
rds_dir <- file.path(results_dir, "rds")
dir.create(rds_dir, recursive = TRUE, showWarnings = FALSE)

dds_path <- file.path(rds_dir, paste0("dds_", strain_tag, "_all_timepoints.rds"))
saveRDS(dds, file = dds_path)

cat("✅ Full DDS saved to:", dds_path, "\n")
cat("   Dimensions       :", nrow(dds), "genes ×", ncol(dds), "samples\n")
cat("   Note             : this holds ALL time points. Script 02 subsets to the chosen one.\n")

</br>

```r
sessionInfo()
```